# Calidad de datos

## Imports

In [42]:
import pandas as pd
import numpy as np
import sqlite3
import json


### Variables globales de conexión

In [4]:
DB_NAME = "openbeauty"
DB_PATH = f"../data/{DB_NAME}.db"
TABLE_NAME = "raw_products"

conn = sqlite3.connect(DB_PATH)


Objetivo: Responder que tan util es esta data ?
* Unicidad
* Concistencia
* Inconsistencia 

In [5]:
df_raw_products = pd.read_sql("SELECT * FROM raw_products", conn)

In [6]:
df_raw_products.head()

,_id,code,rev,update_key,brands,product_name,product_type,countries,countries_tags,countries_hierarchy,...,created_t,last_modified_t,last_updated_t,creator,last_editor,last_modified_by,data_quality_tags,page,batch_id,dtinserted
0,8410757001090,8410757001090,34,key_1743168287,S'nonas,Crema Manos,beauty,"Morocco, United States, en:france","[""en:france"", ""en:morocco"", ""en:united-states""]","[""en:france"", ""en:morocco"", ""en:united-states""]",...,1731329832,1779248127,1779248127,smoothie-app,agenticcommonsbot,agenticcommonsbot,"[""en:no-packaging-data""]",1,1,2026-05-22 16:05:44.417078
1,6281006408647,6281006408647,30,brands,"Unilever,Vaseline",,beauty,"Germany, Morocco, en:saudi-arabia","[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]","[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]",...,1731340361,1771266702,1771266702,smoothie-app,scanbot,scanbot,"[""en:no-packaging-data""]",1,1,2026-05-22 16:05:44.417078
2,80466468,80466468,68,brands,"unilever, dove",DOVE Déodorant Femme Anti-Transpirant Stick Or...,beauty,"France, Libya, Morocco, Saudi Arabia, Spain, e...","[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo...","[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo...",...,1537085263,1779261226,1779261226,openfoodfacts-contributors,agenticcommonsbot,agenticcommonsbot,"[""en:packaging-data-incomplete""]",1,1,2026-05-22 16:05:44.417078
3,3337875598996,3337875598996,33,brands,CeraVe,moisturising cream,beauty,"France,Morocco,Romania","[""en:france"", ""en:morocco"", ""en:romania""]","[""en:france"", ""en:morocco"", ""en:romania""]",...,1730480414,1779263998,1779263998,smoothie-app,agenticcommonsbot,agenticcommonsbot,"[""en:packaging-data-incomplete""]",1,1,2026-05-22 16:05:44.417078
4,3574669909594,3574669909594,35,key_1743168287,Johnson’s baby,,beauty,"Algeria, Morocco, en:ukraine","[""en:algeria"", ""en:morocco"", ""en:ukraine""]","[""en:algeria"", ""en:morocco"", ""en:ukraine""]",...,1660916439,1775820851,1775820851,smoothie-app,foodless,foodless,"[""en:no-packaging-data""]",1,1,2026-05-22 16:05:44.417078


In [7]:
df_raw_products.shape

(50000, 32)

In [8]:
df = df_raw_products

## Data Quality

### Completitud 

In [9]:
df[['countries','countries_tags', 'countries_hierarchy']].isnull().sum()

countries              736
countries_tags         495
countries_hierarchy    736
dtype: int64

Observando que la cantidad de nulos dentro de las variables countries, se verificara sus diferencias para unificarlas

In [10]:
df[['countries','countries_tags', 'countries_hierarchy']].head()

,countries,countries_tags,countries_hierarchy
0,"Morocco, United States, en:france","[""en:france"", ""en:morocco"", ""en:united-states""]","[""en:france"", ""en:morocco"", ""en:united-states""]"
1,"Germany, Morocco, en:saudi-arabia","[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]","[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]"
2,"France, Libya, Morocco, Saudi Arabia, Spain, e...","[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo...","[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo..."
3,"France,Morocco,Romania","[""en:france"", ""en:morocco"", ""en:romania""]","[""en:france"", ""en:morocco"", ""en:romania""]"
4,"Algeria, Morocco, en:ukraine","[""en:algeria"", ""en:morocco"", ""en:ukraine""]","[""en:algeria"", ""en:morocco"", ""en:ukraine""]"


In [11]:
df[['categories', 'categories_hierarchy']].head()

,categories,categories_hierarchy
0,"Creams,Non-food-products,Open-beauty-facts","[""en:Creams"", ""en:Non-food-products"", ""en:Open..."
1,"Incorrect product type,Non-food-products,Open-...","[""en:incorrect-product-type"", ""en:Non-food-pro..."
2,"Déodorants anti-transpirants, en:open-beauty-f...","[""en:D\u00e9odorants anti-transpirants"", ""en:o..."
3,fr:baume hydratant,"[""fr:baume hydratant""]"
4,,[]


In [12]:
df[['categories','categories_hierarchy']].isnull().sum()

categories              18086
categories_hierarchy    18056
dtype: int64

#### Unificando variables que son "iguales"

In [13]:
df['canonical_countries'] = df['countries_hierarchy'].combine_first(df['countries_tags']).combine_first(df['countries'])

In [14]:
df['canonical_categories'] = df['categories_hierarchy'].combine_first(df['categories'])

In [15]:
df.columns

Index(['_id', 'code', 'rev', 'update_key', 'brands', 'product_name',
       'product_type', 'countries', 'countries_tags', 'countries_hierarchy',
       'categories_hierarchy', 'categories', 'product_quantity',
       'product_quantity_unit', 'quantity', 'ingredients_n',
       'known_ingredients_n', 'unknown_ingredients_n', 'completeness',
       'scans_n', 'unique_scans_n', 'popularity_tags', 'created_t',
       'last_modified_t', 'last_updated_t', 'creator', 'last_editor',
       'last_modified_by', 'data_quality_tags', 'page', 'batch_id',
       'dtinserted', 'canonical_countries', 'canonical_categories'],
      dtype='str')

### Umbral de Tolerancia

Definiendo las columnas que determinan la calidad de los datos

In [16]:
quality_columns = [
    "code",
    "product_name",
    "product_type",
    "brands",
    "canonical_categories",
    "canonical_countries",
    "ingredients_n"
]

Definiendo las columnas criticas que determinan la calidad 

In [17]:
critical_columns = [
    "product_name",
    "brands",
    "canonical_categories",
    "canonical_countries"
]

In [18]:
df['completeness_ratio'] = df[quality_columns].notna().mean(axis=1)

In [19]:
df['compl_critical_ratio'] = df[critical_columns].notna().mean(axis=1)

In [20]:
df.columns

Index(['_id', 'code', 'rev', 'update_key', 'brands', 'product_name',
       'product_type', 'countries', 'countries_tags', 'countries_hierarchy',
       'categories_hierarchy', 'categories', 'product_quantity',
       'product_quantity_unit', 'quantity', 'ingredients_n',
       'known_ingredients_n', 'unknown_ingredients_n', 'completeness',
       'scans_n', 'unique_scans_n', 'popularity_tags', 'created_t',
       'last_modified_t', 'last_updated_t', 'creator', 'last_editor',
       'last_modified_by', 'data_quality_tags', 'page', 'batch_id',
       'dtinserted', 'canonical_countries', 'canonical_categories',
       'completeness_ratio', 'compl_critical_ratio'],
      dtype='str')

In [21]:
df.head()

,_id,code,rev,update_key,brands,product_name,product_type,countries,countries_tags,countries_hierarchy,...,last_editor,last_modified_by,data_quality_tags,page,batch_id,dtinserted,canonical_countries,canonical_categories,completeness_ratio,compl_critical_ratio
0,8410757001090,8410757001090,34,key_1743168287,S'nonas,Crema Manos,beauty,"Morocco, United States, en:france","[""en:france"", ""en:morocco"", ""en:united-states""]","[""en:france"", ""en:morocco"", ""en:united-states""]",...,agenticcommonsbot,agenticcommonsbot,"[""en:no-packaging-data""]",1,1,2026-05-22 16:05:44.417078,"[""en:france"", ""en:morocco"", ""en:united-states""]","[""en:Creams"", ""en:Non-food-products"", ""en:Open...",1.0,1.0
1,6281006408647,6281006408647,30,brands,"Unilever,Vaseline",,beauty,"Germany, Morocco, en:saudi-arabia","[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]","[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]",...,scanbot,scanbot,"[""en:no-packaging-data""]",1,1,2026-05-22 16:05:44.417078,"[""en:germany"", ""en:morocco"", ""en:saudi-arabia""]","[""en:incorrect-product-type"", ""en:Non-food-pro...",1.0,1.0
2,80466468,80466468,68,brands,"unilever, dove",DOVE Déodorant Femme Anti-Transpirant Stick Or...,beauty,"France, Libya, Morocco, Saudi Arabia, Spain, e...","[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo...","[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo...",...,agenticcommonsbot,agenticcommonsbot,"[""en:packaging-data-incomplete""]",1,1,2026-05-22 16:05:44.417078,"[""en:algeria"", ""en:france"", ""en:libya"", ""en:mo...","[""en:D\u00e9odorants anti-transpirants"", ""en:o...",1.0,1.0
3,3337875598996,3337875598996,33,brands,CeraVe,moisturising cream,beauty,"France,Morocco,Romania","[""en:france"", ""en:morocco"", ""en:romania""]","[""en:france"", ""en:morocco"", ""en:romania""]",...,agenticcommonsbot,agenticcommonsbot,"[""en:packaging-data-incomplete""]",1,1,2026-05-22 16:05:44.417078,"[""en:france"", ""en:morocco"", ""en:romania""]","[""fr:baume hydratant""]",1.0,1.0
4,3574669909594,3574669909594,35,key_1743168287,Johnson’s baby,,beauty,"Algeria, Morocco, en:ukraine","[""en:algeria"", ""en:morocco"", ""en:ukraine""]","[""en:algeria"", ""en:morocco"", ""en:ukraine""]",...,foodless,foodless,"[""en:no-packaging-data""]",1,1,2026-05-22 16:05:44.417078,"[""en:algeria"", ""en:morocco"", ""en:ukraine""]",[],1.0,1.0


### Duplicidad 

In [22]:
dup_columns = ['code', '_id']

In [23]:
df[dup_columns]

,code,_id
0,8410757001090,8410757001090
1,6281006408647,6281006408647
2,80466468,80466468
3,3337875598996,3337875598996
4,3574669909594,3574669909594
...,...,...
49995,4006000058917,4006000058917
49996,4005808704514,4005808704514
49997,4005900371522,4005900371522
49998,4005900540300,4005900540300


In [24]:
(df['code'] == df['_id']).sum()

np.int64(50000)

De 50000 registros, se calculo la suma cuando code e id, son iguales, el resultado muestra que ambas columnas poseen el mismo valor.

In [25]:
dup_columns = dup_columns[:1] + ['product_name']
dup_columns

['code', 'product_name']

Dado que cod e _id, son iguales, se usara code para la duplicidad ya que este campo,se toma en cuenta para la calidad de columnas

In [26]:
for col in dup_columns:
    df[f"is_duplicate_{col}"] = df[col].duplicated(keep=False).astype(int)

In [27]:
df.columns

Index(['_id', 'code', 'rev', 'update_key', 'brands', 'product_name',
       'product_type', 'countries', 'countries_tags', 'countries_hierarchy',
       'categories_hierarchy', 'categories', 'product_quantity',
       'product_quantity_unit', 'quantity', 'ingredients_n',
       'known_ingredients_n', 'unknown_ingredients_n', 'completeness',
       'scans_n', 'unique_scans_n', 'popularity_tags', 'created_t',
       'last_modified_t', 'last_updated_t', 'creator', 'last_editor',
       'last_modified_by', 'data_quality_tags', 'page', 'batch_id',
       'dtinserted', 'canonical_countries', 'canonical_categories',
       'completeness_ratio', 'compl_critical_ratio', 'is_duplicate_code',
       'is_duplicate_product_name'],
      dtype='str')

### Calidad del campo ingredientes 

In [28]:
df[['ingredients_n', 'known_ingredients_n', 'unknown_ingredients_n']].dtypes

ingredients_n                str
known_ingredients_n      float64
unknown_ingredients_n        str
dtype: object

In [33]:
total_ingredients = (
    pd.to_numeric(df["ingredients_n"], errors="coerce")
    .fillna(0)
    .astype(int)
)
know_ingredients = (
    pd.to_numeric(df["known_ingredients_n"], errors="coerce")
    .fillna(0)
    .astype(int)
)
unknow_ingredients = (
    pd.to_numeric(df["unknown_ingredients_n"], errors="coerce")
    .fillna(0)
    .astype(int)
)


In [34]:
df['ingredients_match'] = (
    total_ingredients == (know_ingredients + unknow_ingredients)
).astype(int)

In [37]:
(df[df['ingredients_match']==0])[['ingredients_n','known_ingredients_n', 'unknown_ingredients_n','ingredients_match']].tail()

,ingredients_n,known_ingredients_n,unknown_ingredients_n,ingredients_match
41239,35.0,10.0,10,0
41241,17.0,14.0,5,0
41243,15.0,10.0,6,0
41249,35.0,27.0,9,0
41371,29.0,6.0,22,0


In [38]:
ingredients_sum = know_ingredients + unknow_ingredients

In [43]:
conditions = [
    df['ingredients_match'] == 1,
    ingredients_sum > total_ingredients,
    ingredients_sum < total_ingredients
]
ingredients_result =  ["Perfect Match", "Sum > total", "Suma< total"]

In [44]:
df['ingredients_discrepancy_type'] = np.select(
    conditions, ingredients_result, default="Error"
)

In [45]:
df['unknown_ingredients_ratio'] = unknow_ingredients.div(total_ingredients)

In [46]:
df['ingredients_quality_score'] = 1 - df["unknown_ingredients_ratio"]

In [47]:
df.columns

Index(['_id', 'code', 'rev', 'update_key', 'brands', 'product_name',
       'product_type', 'countries', 'countries_tags', 'countries_hierarchy',
       'categories_hierarchy', 'categories', 'product_quantity',
       'product_quantity_unit', 'quantity', 'ingredients_n',
       'known_ingredients_n', 'unknown_ingredients_n', 'completeness',
       'scans_n', 'unique_scans_n', 'popularity_tags', 'created_t',
       'last_modified_t', 'last_updated_t', 'creator', 'last_editor',
       'last_modified_by', 'data_quality_tags', 'page', 'batch_id',
       'dtinserted', 'canonical_countries', 'canonical_categories',
       'completeness_ratio', 'compl_critical_ratio', 'is_duplicate_code',
       'is_duplicate_product_name', 'ingredients_match',
       'ingredients_discrepancy_type', 'unknown_ingredients_ratio',
       'ingredients_quality_score'],
      dtype='str')